# Section B – Question 1: CIFAR-10 with 500 training images under distribution shift

| Item | Value |
|---|---|
| Dataset split seed (train_small: 50 images/class; stratified 400 train / 100 val) | 42 |
| Training seeds (mean ± std reported) | 0, 1, 2 |
| shifted_test corruption seed | 2024 |
| Validation-shift seed (model selection only) | 777 |
| Max epochs / parameter limit | 60 / 1,000,000 |

In [ ]:
import os, time, copy, math, random, itertools
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEEDS = [0, 1, 2]           # training seeds (mean +- std)
MAX_EPOCHS = 60             # assignment limit
SPLIT_SEED = 42             # train_small selection and 400/100 split
TEST_CORRUPT_SEED = 2024    # fixed seed of shifted_test
VAL_CORRUPT_SEED = 777      # fixed seed of the validation-shift set (model selection only)

CLASSES = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

print("device:", DEVICE, "| seeds:", SEEDS, "| max epochs:", MAX_EPOCHS,
      "| split seed:", SPLIT_SEED, "| test-corruption seed:", TEST_CORRUPT_SEED)

## 1. Data: train_small (500) → 400 train / 100 val, clean_test, shifted_test

In [ ]:
tr_ds = torchvision.datasets.CIFAR10(root="./data", train=True,  download=True)
te_ds = torchvision.datasets.CIFAR10(root="./data", train=False, download=True)

def to_tensor(ds):
    x = torch.from_numpy(np.asarray(ds.data)).permute(0, 3, 1, 2).float() / 255.0   # (N,3,32,32) in [0,1]
    y = torch.tensor(ds.targets)
    return x, y

Xtr_all, Ytr_all = to_tensor(tr_ds)
Xte, Yte = to_tensor(te_ds)                                  # clean_test (10,000)

# train_small: 50 images per class, then stratified 40 train / 10 val per class
rng = np.random.RandomState(SPLIT_SEED)
train_idx, val_idx = [], []
for c in range(10):
    idx = np.where(Ytr_all.numpy() == c)[0]
    chosen = rng.choice(idx, 50, replace=False)
    train_idx += chosen[:40].tolist()
    val_idx   += chosen[40:].tolist()
assert len(train_idx) == 400 and len(val_idx) == 100 and not set(train_idx) & set(val_idx)

Xtr, Ytr = Xtr_all[train_idx].to(DEVICE), Ytr_all[train_idx].to(DEVICE)
Xval, Yval = Xtr_all[val_idx].to(DEVICE), Ytr_all[val_idx].to(DEVICE)
print("train:", tuple(Xtr.shape), "val:", tuple(Xval.shape), "clean_test:", tuple(Xte.shape))
print("train per class:", torch.bincount(Ytr.cpu()).tolist(), "| val per class:", torch.bincount(Yval.cpu()).tolist())

## 2. Corruption pipeline (test side only)
`shifted_test` = rotation (±25°) → brightness & contrast → ~25 % rectangular occlusion → hidden corruption. A validation-shift set (listed corruptions only, different seed) is built from the 100 validation images and is the only shifted data used for model selection.

In [ ]:
def hidden_corruption(x, gen, p=1.0):
    # Hidden corruption (stand-in until the official one is provided): random grayscale (kind=1) or channel swap (kind=2).
    B, C, H, W = x.shape
    apply = torch.rand(B, generator=gen) < p
    kind_sel = torch.randint(0, 2, (B,), generator=gen)
    perm_id = torch.randint(0, 5, (B,), generator=gen)
    perms = torch.tensor([[0,2,1],[1,0,2],[1,2,0],[2,0,1],[2,1,0]])
    idx = perms[perm_id]
    swapped = torch.gather(x, 1, idx[:, :, None, None].expand(-1, -1, H, W))
    gray = (0.299*x[:,0] + 0.587*x[:,1] + 0.114*x[:,2]).unsqueeze(1).expand(-1, 3, -1, -1)
    alt = torch.where((kind_sel == 0)[:, None, None, None], gray, swapped)
    out = torch.where(apply[:, None, None, None], alt, x)
    kind = torch.where(apply, kind_sel + 1, torch.zeros_like(kind_sel))   # 0 none, 1 gray, 2 swap
    return out, kind

def corrupt(x, seed, hidden=True, hidden_p=0.5):
    gen = torch.Generator().manual_seed(seed)
    B, C, H, W = x.shape
    meta = {}
    # 1) random rotation up to +-25 degrees
    ang = (torch.rand(B, generator=gen) * 2 - 1) * 25.0
    a = ang * math.pi / 180
    theta = torch.zeros(B, 2, 3)
    theta[:,0,0] = torch.cos(a); theta[:,0,1] = -torch.sin(a)
    theta[:,1,0] = torch.sin(a); theta[:,1,1] =  torch.cos(a)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    out = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    # 2) brightness and contrast
    b = 0.5 + torch.rand(B, generator=gen)          # x0.5 .. x1.5
    c = 0.5 + torch.rand(B, generator=gen)
    out = out * b[:, None, None, None]
    m = out.mean(dim=(1, 2, 3), keepdim=True)
    out = ((out - m) * c[:, None, None, None] + m).clamp(0, 1)
    # 3) random rectangular occlusion covering ~25% of the image (black)
    area = 0.25 * H * W
    ratio = torch.exp((torch.rand(B, generator=gen) * 2 - 1) * math.log(2.0))   # aspect 0.5..2
    h = torch.sqrt(area * ratio).clamp(max=H).round().long()
    w = (area / h.float()).clamp(max=W).round().long()
    top  = (torch.rand(B, generator=gen) * (H - h + 1).float()).floor().long()
    left = (torch.rand(B, generator=gen) * (W - w + 1).float()).floor().long()
    ys = torch.arange(H)[None, :, None]; xs = torch.arange(W)[None, None, :]
    mask = (ys >= top[:,None,None]) & (ys < (top+h)[:,None,None]) & (xs >= left[:,None,None]) & (xs < (left+w)[:,None,None])
    out = out * (~mask)[:, None].float()
    # 4) hidden corruption
    kind = torch.zeros(B, dtype=torch.long)
    if hidden:
        out, kind = hidden_corruption(out, gen, p=hidden_p)
    meta.update(angle=ang, brightness=b, contrast=c, occ_frac=mask.float().mean(dim=(1, 2)), hidden=kind)
    return out, meta

Xte_shift_cpu, META_TEST = corrupt(Xte, TEST_CORRUPT_SEED)                 # shifted_test (fixed seed)
Xval_shift_cpu, _        = corrupt(Xtr_all[val_idx], VAL_CORRUPT_SEED, hidden=False)   # selection only
Xte_hid_cpu, KIND_HID    = hidden_corruption(Xte, torch.Generator().manual_seed(TEST_CORRUPT_SEED + 1), p=1.0)  # hidden-only (Part E4)

Xte_shift, Xval_shift, Xte_hid, Xte_dev = Xte_shift_cpu.to(DEVICE), Xval_shift_cpu.to(DEVICE), Xte_hid_cpu.to(DEVICE), Xte.to(DEVICE)
Yte_dev = Yte.to(DEVICE)
print("mean occluded fraction:", META_TEST["occ_frac"].mean().item())

def show_row(imgs, titles=None, row_title=None):
    n = len(imgs); fig, ax = plt.subplots(1, n, figsize=(1.6*n, 1.9))
    for i in range(n):
        ax[i].imshow(imgs[i].permute(1, 2, 0).cpu().numpy().clip(0, 1)); ax[i].axis("off")
        if titles: ax[i].set_title(titles[i], fontsize=8)
    if row_title: fig.suptitle(row_title, fontsize=10, y=1.08)
    plt.show()
show_row(Xte[:8], [CLASSES[i] for i in Yte[:8]], "clean_test samples")
show_row(Xte_shift_cpu[:8], None, "shifted_test samples (same images, corrupted)")

## 3. On-the-fly augmentations (GPU, batched, training data only)

In [ ]:
def _u(B, lo, hi, dev): return torch.rand(B, device=dev) * (hi - lo) + lo
def _gray(x): return (0.299*x[:,0] + 0.587*x[:,1] + 0.114*x[:,2]).unsqueeze(1)

def geometric_aug(x, rot=25.0, trans_px=4, scale=(0.9, 1.1), flip=True):
    # random horizontal flip + random rotation + scale + translation (translation = random-crop-with-padding equivalent)
    B, _, H, W = x.shape; dev = x.device
    if flip:
        m = torch.rand(B, device=dev) < 0.5
        x = torch.where(m[:, None, None, None], x.flip(3), x)
    ang = _u(B, -rot, rot, dev) * math.pi / 180
    z = _u(B, scale[0], scale[1], dev)
    tx = _u(B, -trans_px, trans_px, dev) * 2.0 / W
    ty = _u(B, -trans_px, trans_px, dev) * 2.0 / H
    c, s = torch.cos(ang) / z, torch.sin(ang) / z
    theta = torch.stack([torch.stack([c, -s, tx], 1), torch.stack([s, c, ty], 1)], 1)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    return F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)

def photometric_aug(x, bright=(0.5, 1.5), contrast=(0.5, 1.5), sat=(0.5, 1.5),
                    p_gray=0.15, p_swap=0.15, p_blur=0.3, sigma=(0.3, 1.5), p_noise=0.4, noise_std=0.06):
    B, C, H, W = x.shape; dev = x.device
    x = x * _u(B, *bright, dev)[:, None, None, None]                             # brightness
    m = x.mean(dim=(1, 2, 3), keepdim=True)
    x = (x - m) * _u(B, *contrast, dev)[:, None, None, None] + m                 # contrast
    g = _gray(x)
    x = (g + _u(B, *sat, dev)[:, None, None, None] * (x - g)).clamp(0, 1)        # saturation
    gm = torch.rand(B, device=dev) < p_gray                                      # random grayscale
    x = torch.where(gm[:, None, None, None], _gray(x).expand(-1, 3, -1, -1), x)
    sm = torch.rand(B, device=dev) < p_swap                                      # random channel shuffle
    perm = torch.rand(B, 3, device=dev).argsort(1)
    sw = torch.gather(x, 1, perm[:, :, None, None].expand(-1, -1, H, W))
    x = torch.where(sm[:, None, None, None], sw, x)
    # gaussian blur (per-image sigma, separable, groups conv); non-blurred images get sigma~0 = identity kernel
    sig = torch.where(torch.rand(B, device=dev) < p_blur, _u(B, *sigma, dev), torch.full((B,), 0.05, device=dev))
    r = torch.arange(-3, 4, device=dev).float()[None, :]
    k = torch.exp(-(r ** 2) / (2 * sig[:, None] ** 2)); k = k / k.sum(1, keepdim=True)
    k = k.repeat_interleave(C, 0)                                                # (B*C, 7)
    xx = F.pad(x.reshape(1, B * C, H, W), (3, 3, 3, 3), mode="reflect")
    xx = F.conv2d(xx, k[:, None, None, :], groups=B * C)
    xx = F.conv2d(xx, k[:, None, :, None], groups=B * C)
    x = xx.reshape(B, C, H, W)
    ns = _u(B, 0, noise_std, dev) * (torch.rand(B, device=dev) < p_noise).float() # gaussian noise
    return (x + torch.randn_like(x) * ns[:, None, None, None]).clamp(0, 1)

def cutout_aug(x, p=0.7, area=(0.05, 0.30), ratio=(0.5, 2.0)):
    # random erasing: one rectangle (5-30% of the image) filled black or with a random colour
    B, _, H, W = x.shape; dev = x.device
    apply = torch.rand(B, device=dev) < p
    a = _u(B, *area, dev) * H * W
    r = torch.exp(_u(B, math.log(ratio[0]), math.log(ratio[1]), dev))
    h = torch.sqrt(a * r).clamp(1, H); w = (a / h).clamp(1, W)
    cy = torch.rand(B, device=dev) * H; cx = torch.rand(B, device=dev) * W
    ys = (torch.arange(H, device=dev).float() + 0.5)[None, :, None]
    xs = (torch.arange(W, device=dev).float() + 0.5)[None, None, :]
    mask = ((ys - cy[:, None, None]).abs() < h[:, None, None] / 2) & ((xs - cx[:, None, None]).abs() < w[:, None, None] / 2)
    mask = mask & apply[:, None, None]
    fill = torch.rand(B, 3, 1, 1, device=dev) * (torch.rand(B, 1, 1, 1, device=dev) > 0.5).float()
    return torch.where(mask[:, None], fill.expand_as(x), x)

def mix_batch(x, y, p=0.8, alpha=0.4):
    # Mixup or CutMix (50/50) with soft labels
    if np.random.rand() > p: return x, y
    B, _, H, W = x.shape; dev = x.device
    lam = float(np.random.beta(alpha, alpha)); perm = torch.randperm(B, device=dev)
    if np.random.rand() < 0.5:                                  # mixup
        x = lam * x + (1 - lam) * x[perm]
    else:                                                       # cutmix
        cr = math.sqrt(1 - lam); ch, cw = int(H * cr), int(W * cr)
        cy, cx = np.random.randint(H), np.random.randint(W)
        y1, y2 = max(cy - ch // 2, 0), min(cy + ch // 2, H); x1, x2 = max(cx - cw // 2, 0), min(cx + cw // 2, W)
        xp = x[perm]; x = x.clone(); x[:, :, y1:y2, x1:x2] = xp[:, :, y1:y2, x1:x2]
        lam = 1 - ((y2 - y1) * (x2 - x1)) / (H * W)
    return x, lam * y + (1 - lam) * y[perm]

def augment(x, y_soft, cfg, hp):
    if cfg["geo"]:    x = geometric_aug(x, rot=hp["rot"])
    if cfg["photo"]:  x = photometric_aug(x)
    if cfg["cutout"]: x = cutout_aug(x)
    if cfg["mix"]:    x, y_soft = mix_batch(x, y_soft)
    return x, y_soft

## 4. Model: stem → 3 Inception modules (parallel 1×1 / 3×3 / 5×5 branches with 1×1 reductions + pool branch) → GAP → FC head

In [ ]:
class ConvBNReLU(nn.Sequential):
    def __init__(self, i, o, k):
        super().__init__(nn.Conv2d(i, o, k, padding=k // 2, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))

class Inception(nn.Module):
    def __init__(self, i, c1, c3r, c3, c5r, c5, cp):
        super().__init__()
        self.b1 = ConvBNReLU(i, c1, 1)
        self.b3 = nn.Sequential(ConvBNReLU(i, c3r, 1), ConvBNReLU(c3r, c3, 3))
        self.b5 = nn.Sequential(ConvBNReLU(i, c5r, 1), ConvBNReLU(c5r, c5, 5))
        self.bp = nn.Sequential(nn.MaxPool2d(3, 1, 1), ConvBNReLU(i, cp, 1))
        self.out_ch = c1 + c3 + c5 + cp
    def forward(self, x):
        return torch.cat([self.b1(x), self.b3(x), self.b5(x), self.bp(x)], 1)

class Net(nn.Module):
    def __init__(self, dropout=0.5):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std",  torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.stem = nn.Sequential(ConvBNReLU(3, 32, 3), ConvBNReLU(32, 64, 3), nn.MaxPool2d(2))        # 16x16
        self.inc1 = Inception(64, 16, 24, 32, 8, 16, 16)                                               # -> 80 ch
        self.inc2 = Inception(80, 32, 32, 48, 12, 24, 24)                                              # -> 128 ch (after pool: 8x8)
        self.inc3 = Inception(128, 48, 48, 64, 16, 32, 32)                                             # -> 176 ch (after pool: 4x4)
        self.pool = nn.MaxPool2d(2)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(176, 128), nn.ReLU(inplace=True),
                                  nn.Dropout(dropout), nn.Linear(128, 10))                             # dropout only in the FC head
    def forward(self, x):
        x = (x - self.mean) / self.std
        x = self.inc1(self.stem(x))
        x = self.inc2(self.pool(x))
        x = self.inc3(self.pool(x))
        x = F.adaptive_avg_pool2d(x, 1)
        return self.head(x)

n_params = sum(p.numel() for p in Net().parameters() if p.requires_grad)
assert n_params <= 1_000_000
print(f"trainable parameters: {n_params:,}  (limit 1,000,000)")

## 5. Training / evaluation utilities
Optimiser: SGD + Nesterov momentum, **OneCycle** LR schedule, weight decay, BatchNorm, label smoothing (when stated in the config). Model selection per run = **best checkpoint** on the validation split (mean of clean-val and validation-shift accuracy) with early-stopping patience. Test sets are evaluated once, after training, with the selected checkpoint.

In [ ]:
HP_DEFAULT = dict(lr=0.05, wd=5e-4, dropout=0.5, rot=25.0, bs=50)
N1, N2, N3 = "No augmentation", "Geometric only (crop, flip, rotation)", "Photometric only (color jitter, blur, noise)"
N4, N5, N6, N7 = "Geometric + photometric", "+ Cutout / Random Erasing", "+ Mixup/CutMix + label smoothing", "Final model"
CONFIGS = {
    N1: dict(geo=False, photo=False, cutout=False, mix=False, ls=0.0),
    N2: dict(geo=True,  photo=False, cutout=False, mix=False, ls=0.0),
    N3: dict(geo=False, photo=True,  cutout=False, mix=False, ls=0.0),
    N4: dict(geo=True,  photo=True,  cutout=False, mix=False, ls=0.0),
    N5: dict(geo=True,  photo=True,  cutout=True,  mix=False, ls=0.0),
    N6: dict(geo=True,  photo=True,  cutout=True,  mix=True,  ls=0.1),
}

@torch.no_grad()
def predict_probs(model, X, bs=2000):
    model.eval()
    return torch.cat([F.softmax(model(X[i:i + bs]), 1) for i in range(0, len(X), bs)])

def accuracy(model, X, Y): return (predict_probs(model, X).argmax(1) == Y).float().mean().item()

def run(cfg, hp, seed, epochs=None, patience=30):
    set_seed(seed)
    epochs = epochs or MAX_EPOCHS
    model = Net(hp["dropout"]).to(DEVICE)
    opt = torch.optim.SGD(model.parameters(), lr=hp["lr"], momentum=0.9, nesterov=True, weight_decay=hp["wd"])
    spe = len(Xtr) // hp["bs"]
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=hp["lr"], total_steps=epochs * spe, pct_start=0.2)
    hist = {k: [] for k in ["train_loss", "train_acc", "val_acc", "val_loss", "val_shift_acc"]}
    best, best_state, best_ep, bad = -1, None, 0, 0
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xtr), device=DEVICE); tl = 0.0
        for i in range(spe):
            idx = perm[i * hp["bs"]:(i + 1) * hp["bs"]]
            x = Xtr[idx]; y = F.one_hot(Ytr[idx], 10).float()
            y = y * (1 - cfg["ls"]) + cfg["ls"] / 10                      # label smoothing
            x, y = augment(x, y, cfg, hp)                                  # ON-THE-FLY augmentation, training batch only
            loss = -(y * F.log_softmax(model(x), 1)).sum(1).mean()
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step(); tl += loss.item()
        pv = predict_probs(model, Xval)
        va = (pv.argmax(1) == Yval).float().mean().item()
        vl = -torch.log(pv[torch.arange(len(Yval)), Yval] + 1e-8).mean().item()
        vs = accuracy(model, Xval_shift, Yval)
        hist["train_loss"].append(tl / spe); hist["train_acc"].append(accuracy(model, Xtr, Ytr))
        hist["val_acc"].append(va); hist["val_loss"].append(vl); hist["val_shift_acc"].append(vs)
        score = 0.5 * (va + vs)
        if score >= best:
            best, best_ep, bad = score, ep, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break                                      # early stopping
    model.load_state_dict(best_state)
    return dict(seed=seed, best_epoch=best_ep + 1, epochs_run=len(hist["val_acc"]), hist=hist, state=best_state,
                train_acc=accuracy(model, Xtr, Ytr), val_acc=accuracy(model, Xval, Yval),
                val_shift_acc=accuracy(model, Xval_shift, Yval),
                clean_test=accuracy(model, Xte_dev, Yte_dev), shifted_test=accuracy(model, Xte_shift, Yte_dev),
                hidden_only=accuracy(model, Xte_hid, Yte_dev), n_params=sum(p.numel() for p in model.parameters()))

def ms(runs, key):
    v = np.array([r[key] for r in runs]) * 100
    return v.mean(), (v.std(ddof=1) if len(v) > 1 else 0.0)
def fmt(runs, key): m, s = ms(runs, key); return f"{m:.2f} ± {s:.2f}"

def _pad(runs, key):
    L = max(len(r["hist"][key]) for r in runs); a = np.full((len(runs), L), np.nan)
    for i, r in enumerate(runs): a[i, :len(r["hist"][key])] = r["hist"][key]
    return a
def plot_curves(runs, title):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    for key, lab, col in [("train_acc", "train (clean, eval mode)", "tab:blue"), ("val_acc", "validation (clean)", "tab:orange"),
                          ("val_shift_acc", "validation (shifted)", "tab:green")]:
        a = _pad(runs, key) * 100; m, s = np.nanmean(a, 0), np.nanstd(a, 0); ep = np.arange(1, len(m) + 1)
        ax[0].plot(ep, m, label=lab, color=col); ax[0].fill_between(ep, m - s, m + s, color=col, alpha=0.2)
    ax[0].set_title(f"{title} – accuracy (mean ± std over {len(runs)} seeds)"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("accuracy (%)"); ax[0].legend(); ax[0].grid(alpha=0.3)
    for key, lab, col in [("train_loss", "train loss (augmented batches)", "tab:blue"), ("val_loss", "validation loss (clean)", "tab:orange")]:
        a = _pad(runs, key); m, s = np.nanmean(a, 0), np.nanstd(a, 0); ep = np.arange(1, len(m) + 1)
        ax[1].plot(ep, m, label=lab, color=col); ax[1].fill_between(ep, m - s, m + s, color=col, alpha=0.2)
    ax[1].set_title(f"{title} – loss"); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("cross-entropy"); ax[1].legend(); ax[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

R = {}   # results: config name -> list of per-seed run dicts

# Part A – Baseline (no augmentation)
Same network, same regularizers (BatchNorm, dropout, weight decay), trained on the 400 images with **no augmentation**.

In [ ]:
R[N1] = [run(CONFIGS[N1], HP_DEFAULT, s) for s in SEEDS]
print(f"{'seed':>4} {'best_ep':>7} {'train':>7} {'val':>7} {'clean_test':>11} {'shifted_test':>13}")
for r in R[N1]:
    print(f"{r['seed']:>4} {r['best_epoch']:>7} {100*r['train_acc']:7.2f} {100*r['val_acc']:7.2f} {100*r['clean_test']:11.2f} {100*r['shifted_test']:13.2f}")
print("\nBASELINE (mean ± std over seeds)")
for k, n in [("train_acc","train"),("val_acc","validation"),("clean_test","clean_test"),("shifted_test","shifted_test")]:
    print(f"  {n:13s}: {fmt(R[N1], k)} %")
plot_curves(R[N1], "Baseline (no augmentation)")
tr_m, cl_m, sh_m = ms(R[N1],"train_acc")[0], ms(R[N1],"clean_test")[0], ms(R[N1],"shifted_test")[0]
print(f"\ngeneralisation gap (train - clean_test) = {tr_m-cl_m:.1f} pts  -> overfitting indicator")
print(f"shift gap          (clean_test - shifted_test) = {cl_m-sh_m:.1f} pts  -> covariate-shift indicator")

In [ ]:
from IPython.display import Markdown, display
vm = ms(R[N1], "val_acc")[0]
over, shift = (tr_m - cl_m) > 15, (cl_m - sh_m) > 10
kind = "both overfitting and covariate shift" if over and shift else ("overfitting" if over else ("covariate shift" if shift else "no strong failure"))
diag_text = (f"The baseline reaches {tr_m:.1f}% training accuracy but only {vm:.1f}% validation and {cl_m:.1f}% clean-test accuracy "
             f"(train-to-clean gap {tr_m-cl_m:.1f} points): it memorised the 400 images, which is overfitting. On shifted_test accuracy falls further to {sh_m:.1f}% "
             f"(clean-to-shifted gap {cl_m-sh_m:.1f} points): the labels are unchanged but the input distribution moved (rotation, brightness/contrast, occlusion, "
             f"colour corruption), which is covariate shift. Diagnosis: {kind}.")
display(Markdown("**Part A diagnosis:** " + diag_text))

# Part B – Augmentation strategy

| Augmentation | Why (which test corruption / which invariance) |
|---|---|
| Random rotation ±25° (hp `rot`), scale 0.9–1.1, translation ±4 px (= random crop with padding) | Directly matches the test rotation; scale/translation teach position tolerance cheaply. Black corners appear in both train and test. |
| Horizontal flip | Label-preserving for all 10 classes (a flipped cat is still a cat); doubles effective data. |
| Colour jitter: brightness ×0.5–1.5, contrast ×0.5–1.5, saturation, random grayscale (p=.15), channel shuffle (p=.15) | Matches brightness/contrast corruption and prepares for colour-type hidden corruptions *without* knowing it (generalises the idea, does not copy it). |
| Gaussian blur, Gaussian noise | Cheap robustness to sensor-like degradations; reduces reliance on fine high-frequency texture that a 400-image model memorises. |
| Cutout / Random Erasing (5–30 % area, black or random colour) | Matches the ~25 % rectangular occlusion: forces the net to use several object parts instead of one memorised patch. |
| Mixup / CutMix (+ label smoothing 0.1) | Strong regulariser for tiny datasets: soft labels reduce over-confidence and make decision boundaries smoother; CutMix also imitates occlusion. |

**Q1. Would vertical flip help or hurt?** It would most likely **hurt**. CIFAR-10 objects have a canonical upright orientation (sky above, wheels/legs below; ships and planes with sky on top; animals standing). Upside-down images never occur in the clean or shifted test sets (rotation is only ±25°), so vertical flip trains on an unrealistic distribution, wastes the capacity of a small model, and blurs class-specific layout cues. It is a valid label-preserving transform only in datasets without a gravity direction (aerial/microscope images).

**Q2. Rotation up to ±90° while the test only has ±25°?** *Benefit:* covers the test range with margin and gives robustness if the test range were underestimated. *Cost:* most training samples (≈ 72 % lie outside ±25°) show orientations that never appear at test time; the model spends limited capacity on solving a harder task, learns blurrier decision boundaries for upright images, and sees more interpolation artefacts / black corners. It is a bias–variance trade-off: augmentation range should be slightly above the test range (here 25–30°), not far above it. (Tunable via `hp["rot"]`; Part C searches 15° vs 30° on validation only.)

**Q3. Why can too much augmentation reduce accuracy on a small model?** (i) *Capacity*: our network has ~150 k parameters; if the training distribution becomes much harder/wider than the test distribution, the model underfits (training loss stays high). (ii) *Label damage*: stacking occlusion + blur + noise + mixup can erase the class-defining information, so labels become effectively noisy. (iii) *Distribution mismatch*: the model optimises for corruptions that never occur at test time and loses accuracy on the clean-looking part. (iv) With a hard 60-epoch cap, heavy augmentation needs more epochs to converge. Therefore strength is tuned on validation data.

In [ ]:
set_seed(0)
xs = Xval[:8]; dummy = F.one_hot(Yval[:8], 10).float()
show_row(xs, [CLASSES[i] for i in Yval[:8]], "original validation images")
show_row(geometric_aug(xs), None, "geometric (flip + rotation ±25° + scale + translation)")
show_row(photometric_aug(xs), None, "photometric (jitter, grayscale/shuffle, blur, noise)")
show_row(cutout_aug(photometric_aug(geometric_aug(xs)), p=1.0), None, "geometric + photometric + cutout")
mx, _ = mix_batch(cutout_aug(photometric_aug(geometric_aug(xs))), dummy, p=1.0)
show_row(mx, None, "…+ mixup/cutmix (labels become soft)")

# Part C – Regularization and training design (all choices justified)

| Technique | Placement / value | Justification |
|---|---|---|
| **BatchNorm** | after *every* convolution (before ReLU), also inside each Inception branch | Stabilises training with tiny batches/data, allows larger LR, and acts as a mild regulariser (batch noise). |
| **Dropout** | only in the FC head, between the 128-unit hidden layer and the classifier | The FC layer holds the most "memorisation" capacity relative to its use; dropout there decorrelates features. Convolutional feature maps are spatially correlated, so element-wise dropout between early convs removes little information (neighbours compensate) while destroying low-level edge/colour features the small model needs. |
| **Weight decay** (L2, via SGD) | all weights, searched in {5e-4, 5e-3} | Keeps weights small → smoother functions; essential when n=400. |
| **LR schedule** | OneCycle (warm-up 20 % then cosine decay) | Warm-up avoids early divergence with strong augmentation; annealing gives a flat final minimum. |
| **Label smoothing 0.1** | with Mixup/CutMix configs | Prevents over-confident logits, improves calibration and robustness under shift. |
| **Early stopping / checkpoint selection** | patience 30, best validation checkpoint (clean-val + validation-shift) | Guards against late over-fitting; uses the validation split only. |
| **Global average pooling** | instead of large FC layers | Removes most parameters (total ≈ 150 k, limit 1 M) and gives translation tolerance. |

**Hyper-parameter search (validation split only, seed 0, full augmentation config):** grid over learning rate, weight decay, dropout and rotation range. Score = mean of clean-validation and validation-shift accuracy. The test sets are *not* touched for selection (they are only evaluated once per finished run for reporting).

In [ ]:
grid = list(itertools.product([0.03, 0.1], [5e-4, 5e-3], [0.3, 0.5], [15.0, 30.0]))   # lr, wd, dropout, rot
search = []
for lr, wd, do, rot in grid:
    hp = dict(HP_DEFAULT, lr=lr, wd=wd, dropout=do, rot=rot)
    r = run(CONFIGS[N6], hp, seed=0)
    score = 0.5 * (r["val_acc"] + r["val_shift_acc"])
    search.append(dict(lr=lr, wd=wd, dropout=do, rot=rot, val_clean=100*r["val_acc"], val_shift=100*r["val_shift_acc"], score=100*score))
    print(f"lr={lr:<5} wd={wd:<6} dropout={do} rot={rot:<4} -> val_clean {100*r['val_acc']:.1f} | val_shift {100*r['val_shift_acc']:.1f} | score {100*score:.1f}")
sdf = pd.DataFrame(search).sort_values("score", ascending=False).reset_index(drop=True)
display(sdf)
b = sdf.iloc[0]
HP_FINAL = dict(HP_DEFAULT, lr=float(b.lr), wd=float(b.wd), dropout=float(b.dropout), rot=float(b.rot))
print("\nSelected hyper-parameters (validation only):", HP_FINAL)

# Part D – Ablation study (3 seeds each; mean ± std)
Rows 1–6 use the default hyper-parameters (so that only the augmentation changes). The **Final model** = last augmentation recipe with the hyper-parameters selected in Part C.

In [ ]:
for name in [N2, N3, N4, N5, N6]:
    R[name] = [run(CONFIGS[name], HP_DEFAULT, s) for s in SEEDS]
    print(f"{name:45s} shifted_test = {fmt(R[name], 'shifted_test')}")
R[N7] = [run(CONFIGS[N6], HP_FINAL, s) for s in SEEDS]
print(f"{N7:45s} shifted_test = {fmt(R[N7], 'shifted_test')}")

order = [N1, N2, N3, N4, N5, N6, N7]
abl = pd.DataFrame([{"Configuration": n, "Val Acc": fmt(R[n], "val_acc"), "Clean Test": fmt(R[n], "clean_test"),
                     "Shifted Test": fmt(R[n], "shifted_test"), "Hidden-only (extra)": fmt(R[n], "hidden_only"),
                     "Val-shift (extra)": fmt(R[n], "val_shift_acc")} for n in order])
display(abl)

fig, ax = plt.subplots(figsize=(11, 4.8))
m = [ms(R[n], "shifted_test")[0] for n in order]; s = [ms(R[n], "shifted_test")[1] for n in order]
c = [ms(R[n], "clean_test")[0] for n in order]
x = np.arange(len(order))
ax.bar(x - 0.2, c, 0.4, label="clean test", color="tab:blue", alpha=0.7)
ax.bar(x + 0.2, m, 0.4, yerr=s, capsize=4, label="shifted test", color="tab:red", alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels([n.replace(" (", "\n(") for n in order], rotation=25, ha="right", fontsize=8)
ax.set_ylabel("accuracy (%)"); ax.set_title(f"Ablation: accuracy by configuration (mean ± std over {len(SEEDS)} seeds)")
ax.legend(); ax.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()

mu = {n: ms(R[n], "shifted_test")[0] for n in order}
gains = {"Geometric augmentation (vs none)": mu[N2] - mu[N1],
         "Photometric augmentation (vs none)": mu[N3] - mu[N1],
         "Combining geometric + photometric (vs best single)": mu[N4] - max(mu[N2], mu[N3]),
         "Adding Cutout / Random Erasing": mu[N5] - mu[N4],
         "Adding Mixup/CutMix + label smoothing": mu[N6] - mu[N5],
         "Hyper-parameter tuning (Final vs row 6)": mu[N7] - mu[N6]}
print("Shifted-test gain of each single change (percentage points):")
for k, v in gains.items(): print(f"  {v:+6.2f}  {k}")
BEST_CHANGE = max(gains, key=gains.get)
print("\n>> LARGEST single gain:", BEST_CHANGE, f"({gains[BEST_CHANGE]:+.2f} pts)")

In [ ]:
MAP = {"Geometric augmentation (vs none)": "the random rotation of up to ±25° (together with the position and scale variation it brings)",
       "Photometric augmentation (vs none)": "the brightness/contrast changes (and, through grayscale and channel-shuffle, the hidden colour corruption)",
       "Combining geometric + photometric (vs best single)": "the fact that the shifted test applies rotation and brightness/contrast together, so both invariances are needed at once",
       "Adding Cutout / Random Erasing": "the ~25 % rectangular occlusion, which forces the network to classify from partial evidence",
       "Adding Mixup/CutMix + label smoothing": "the over-confidence problem under shift (soft labels), with CutMix also imitating the occlusion",
       "Hyper-parameter tuning (Final vs row 6)": "better regularisation strength (learning rate, weight decay, dropout, rotation range) chosen on validation data"}
g = gains[BEST_CHANGE]
if g > 0:
    abl_text = (f"The largest gain came from '{BEST_CHANGE}' (+{g:.2f} points on shifted test). This matches {MAP[BEST_CHANGE]} in the shifted test. "
                f"Geometric-only reached {mu[N2]:.2f}% and photometric-only {mu[N3]:.2f}% versus {mu[N1]:.2f}% with no augmentation; "
                f"the final model reached {mu[N7]:.2f}% (total gain {mu[N7]-mu[N1]:+.2f} points).")
else:
    abl_text = (f"No single step improved shifted-test accuracy over the previous row (best step: '{BEST_CHANGE}', {g:+.2f} points). "
                f"Accuracy went from {mu[N1]:.2f}% (no augmentation) to {mu[N7]:.2f}% (final model).")
display(Markdown("**Largest gain:** " + abl_text))

## Learning curves of the final model

In [ ]:
plot_curves(R[N7], "Final model")
print("Final model per-seed:", [(r["seed"], round(100*r["shifted_test"], 2)) for r in R[N7]])
print("Selected-epoch per seed:", [r["best_epoch"] for r in R[N7]])

# Part E – Error analysis (final model, seed 0 unless stated)

In [ ]:
M = Net(HP_FINAL["dropout"]).to(DEVICE); M.load_state_dict(R[N7][0]["state"]); M.eval()
probs = predict_probs(M, Xte_shift); pred = probs.argmax(1).cpu().numpy(); yt = Yte.numpy()
conf = probs.max(1).values.cpu().numpy()

# E1: per-class accuracy + normalized confusion matrix
pc = np.array([(pred[yt == c] == c).mean() for c in range(10)]) * 100
pcdf = pd.DataFrame({"class": CLASSES, "shifted_test_acc_%": pc.round(2)}); display(pcdf)
fig, ax = plt.subplots(figsize=(8, 3.8)); ax.bar(CLASSES, pc, color="tab:purple")
ax.set_title("Per-class accuracy on shifted_test (final model, seed 0)"); ax.set_xlabel("class"); ax.set_ylabel("accuracy (%)")
plt.xticks(rotation=40, ha="right"); ax.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()

cm = confusion_matrix(yt, pred, normalize="true")
fig, ax = plt.subplots(figsize=(8, 7)); im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(10)); ax.set_yticks(range(10)); ax.set_xticklabels(CLASSES, rotation=45, ha="right"); ax.set_yticklabels(CLASSES)
for i in range(10):
    for j in range(10): ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=7, color="white" if cm[i, j] > 0.5 else "black")
ax.set_title("Normalized confusion matrix – shifted_test (final model, seed 0)"); ax.set_xlabel("predicted class"); ax.set_ylabel("true class")
plt.colorbar(im, label="fraction of true class"); plt.tight_layout(); plt.show()

# E2: most confused pairs
off = [(cm[i, j], CLASSES[i], CLASSES[j]) for i in range(10) for j in range(10) if i != j]
print("Most confused (true -> predicted, fraction of the true class):")
for v, a, b in sorted(off, reverse=True)[:6]: print(f"  {a:>10s} -> {b:<10s} {v:.3f}")

In [ ]:
def pair_reason(a, b):
    s = {a, b}
    if s == {"cat", "dog"}: return "similar fur texture, pose and body size; the distinguishing details (ears, snout) are exactly what occlusion and blur remove first"
    if s == {"automobile", "truck"}: return "both are boxy road vehicles whose shape differences are subtle at 32x32 pixels"
    if s <= {"airplane", "ship", "bird"}: return "shared sky/water backgrounds and the colour cue being weakened by brightness or grayscale changes"
    if s == {"deer", "horse"}: return "similar four-legged silhouettes on grass/brown backgrounds"
    return "similar shapes, textures or backgrounds at 32x32 resolution, amplified by occlusion and colour/brightness changes"
top3 = sorted(off, reverse=True)[:3]
e2_text = ("Most confused pairs (true -> predicted, share of the true class): " +
           "; ".join(f"{a} -> {b} ({v:.2f}): {pair_reason(a, b)}" for v, a, b in top3) +
           f". Best class: {CLASSES[int(pc.argmax())]} ({pc.max():.1f}%); weakest class: {CLASSES[int(pc.argmin())]} ({pc.min():.1f}%).")
display(Markdown("**E2 – most confused classes:** " + e2_text))

In [ ]:
# E3: >= 8 misclassified images, with the corruption parameters that were applied to each
wrong = np.where(pred != yt)[0]
order_w = wrong[np.argsort(-conf[wrong])]            # most confident mistakes first
picked, seen = [], set()
for i in order_w:
    if yt[i] not in seen: picked.append(i); seen.add(yt[i])
    if len(picked) == 8: break
for i in order_w:
    if len(picked) >= 8: break
    if i not in picked: picked.append(i)
HN = {0: "none", 1: "grayscale", 2: "channel swap"}
fig, ax = plt.subplots(2, 8, figsize=(18, 5.2))
rows = []
for k, i in enumerate(picked):
    ax[0, k].imshow(Xte_shift_cpu[i].permute(1, 2, 0).numpy()); ax[0, k].axis("off")
    ax[0, k].set_title(f"true: {CLASSES[yt[i]]}\npred: {CLASSES[pred[i]]} ({conf[i]:.2f})", fontsize=8)
    ax[1, k].imshow(Xte[i].permute(1, 2, 0).numpy()); ax[1, k].axis("off")
    ang, br, ct = META_TEST["angle"][i].item(), META_TEST["brightness"][i].item(), META_TEST["contrast"][i].item()
    hk = HN[META_TEST["hidden"][i].item()]
    obs = []
    if abs(ang) > 15: obs.append(f"strong rotation ({ang:+.0f}°)")
    if br < 0.75: obs.append("dark image")
    elif br > 1.25: obs.append("over-bright image")
    if ct < 0.75: obs.append("low contrast")
    if META_TEST["hidden"][i].item() == 1: obs.append("colour removed (grayscale)")
    if META_TEST["hidden"][i].item() == 2: obs.append("colour channels swapped")
    obs.append(f"{100*META_TEST['occ_frac'][i].item():.0f}% occluded")
    rows.append(dict(idx=int(i), true=CLASSES[yt[i]], pred=CLASSES[pred[i]], conf=round(float(conf[i]), 2), angle=round(ang, 1),
                     brightness=round(br, 2), contrast=round(ct, 2), hidden=hk, observations="; ".join(obs)))
fig.suptitle("E3: misclassified shifted_test images (top row = what the model saw, bottom row = original clean image)", y=1.0)
plt.tight_layout(); plt.show()
pd.set_option("display.max_colwidth", 120); display(pd.DataFrame(rows))

In [ ]:
lines = []
for r in rows:
    pr = {r["true"], r["pred"]}
    why = pair_reason(r["true"], r["pred"]) if (pr in [{"cat","dog"},{"automobile","truck"},{"deer","horse"}] or (len(pr) == 2 and pr <= {"airplane","ship","bird"})) \
          else "the occluder hides the key object parts so the remaining evidence points to the wrong class"
    lines.append(f"- **Image {r['idx']} ({r['true']} -> {r['pred']}, confidence {r['conf']}):** {r['observations']}. Likely cause: {why}.")
display(Markdown("**E3 – failure explanations:**\n\n" + "\n".join(lines)))

In [ ]:
# E4: hidden-corruption-only test (applied to the clean test set)
print("Hidden-only accuracy (clean test + hidden corruption on every image), mean ± std over seeds:")
for n in [N1, N4, N5, N6, N7]:
    print(f"  {n:45s} clean {fmt(R[n], 'clean_test')}  ->  hidden-only {fmt(R[n], 'hidden_only')}")
ph = predict_probs(M, Xte_hid).argmax(1).cpu().numpy()
print("\nFinal model (seed 0) accuracy by hidden-corruption type on clean test:")
for kd in [1, 2]:
    mk = (KIND_HID.numpy() == kd); print(f"  {HN[kd]:13s}: {100*(ph[mk] == yt[mk]).mean():.2f}%  (n={mk.sum()})")

In [ ]:
cb, hb = ms(R[N1], "clean_test")[0], ms(R[N1], "hidden_only")[0]
cf, hf = ms(R[N7], "clean_test")[0], ms(R[N7], "hidden_only")[0]
acc_g = 100 * (ph[KIND_HID.numpy() == 1] == yt[KIND_HID.numpy() == 1]).mean()
acc_s = 100 * (ph[KIND_HID.numpy() == 2] == yt[KIND_HID.numpy() == 2]).mean()
drop_b, drop_f = cb - hb, cf - hf
verdict = ("generalised well" if (hf - hb) > 5 and drop_f < drop_b else
           "generalised partly" if hf > hb else "did not generalise")
weak = "channel swap" if acc_s < acc_g else "grayscale"
e4_text = (f"On the clean test set with only the hidden corruption applied, the baseline falls from {cb:.1f}% to {hb:.1f}% (drop {drop_b:.1f} points) while the final model "
           f"falls from {cf:.1f}% to {hf:.1f}% (drop {drop_f:.1f} points). The augmentation {verdict}, although it never saw this corruption: the colour jitter, random grayscale "
           f"and channel-shuffle augmentations teach reduced reliance on exact colour cues. By type (final model, seed 0): grayscale {acc_g:.1f}%, channel swap {acc_s:.1f}%; "
           f"the harder type is {weak}.")
display(Markdown("**E4 – hidden-corruption test:** " + e4_text))

# Part F – Test-Time Augmentation (bonus)
Average softmax over N deterministic augmented copies (horizontal flip + ±2 px shifts).

In [ ]:
def shift_img(x, dx, dy):
    if dx == 0 and dy == 0: return x
    p = max(abs(dx), abs(dy)); H, W = x.shape[2:]
    xp = F.pad(x, (p, p, p, p)); return xp[:, :, p + dy:p + dy + H, p + dx:p + dx + W]
TTA_SET = [(0, 0, 0), (1, 0, 0), (0, 2, 0), (0, -2, 0), (0, 0, 2), (0, 0, -2), (1, 2, 2), (1, -2, -2)]   # (flip, dx, dy)

@torch.no_grad()
def tta_probs(model, X, n):
    model.eval(); acc = 0
    for f, dx, dy in TTA_SET[:n]:
        xa = X.flip(3) if f else X
        acc = acc + torch.cat([F.softmax(model(shift_img(xa[i:i + 2000], dx, dy)), 1) for i in range(0, len(X), 2000)])
    return acc / n

def timed(fn):
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    t = time.time(); out = fn()
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    return out, time.time() - t

res = {n: [] for n in [1, 2, 4, 8]}; times = {}
for r in R[N7]:
    mdl = Net(HP_FINAL["dropout"]).to(DEVICE); mdl.load_state_dict(r["state"])
    tta_probs(mdl, Xte_shift[:500], 8)      # warm-up
    for n in res:
        p, t = timed(lambda: tta_probs(mdl, Xte_shift, n))
        res[n].append(100 * (p.argmax(1) == Yte_dev).float().mean().item())
        if r["seed"] == SEEDS[0]: times[n] = t
rows = []
for n in res:
    a = np.array(res[n]); rows.append({"N copies": n, "shifted acc (%)": f"{a.mean():.2f} ± {(a.std(ddof=1) if len(a) > 1 else 0):.2f}",
        "change vs N=1 (pts)": f"{a.mean() - np.mean(res[1]):+.2f}", "inference time for 10k imgs (s)": round(times[n], 3),
        "relative cost": f"{times[n] / times[1]:.1f}x"})
tta_df = pd.DataFrame(rows); display(tta_df)
fig, ax = plt.subplots(figsize=(6.5, 4)); ax.errorbar(list(res), [np.mean(res[n]) for n in res], yerr=[np.std(res[n]) for n in res], marker="o", capsize=4)
ax.set_title("TTA: shifted-test accuracy vs number of augmented copies"); ax.set_xlabel("N copies per image"); ax.set_ylabel("shifted-test accuracy (%)"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
gain8 = np.mean(res[8]) - np.mean(res[1]); cost8 = times[8] / times[1]
tta_text = (f"Averaging the softmax over N=8 flipped/shifted copies changes shifted-test accuracy by {gain8:+.2f} points "
            f"({np.mean(res[1]):.2f}% -> {np.mean(res[8]):.2f}%) at {cost8:.1f}x the inference cost ({times[1]:.3f}s -> {times[8]:.3f}s per 10,000 images).")
display(Markdown("**Part F – TTA result:** " + tta_text))